chargement

In [ ]:
import os #dossiers
import json
import random #operations
import hashlib #
import numpy as np

from collections import Counter
from tqdm import tqdm #barre de progression
from PIL import Image
from pycocotools.coco import COCO

# Vérifier le dossier du projet
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")

print("Project directory:", os.getcwd())

random.seed(42)
np.random.seed(42)

# Charger les annotations COCO
coco = COCO("data/annotations/captions_train2017.json")

print("Imports et annotations chargés avec succès.")

Project directory: c:\Users\haman\Downloads\Telegram Desktop\image_pro
loading annotations into memory...
Done (t=1.15s)
creating index...
index created!
Imports et annotations chargés avec succès.


In [ ]:
with open("splits/splits_all.json", "r", encoding="utf-8") as f:
    splits = json.load(f)

kb_ids = splits["kb"]
val_ids = splits["val"]
test_ids = splits["test"]
#load the test strata
test_strata = splits["test_strata"]

print("Knowledge Base:", len(kb_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Knowledge Base: 10000
Validation: 100
Test: 100


In [ ]:
# Vérifier les tailles
assert len(kb_ids) == 10000
assert len(val_ids) == 100
assert len(test_ids) == 100

# La validation est un sous-ensemble de la KB   # a courigé
assert set(val_ids).issubset(set(kb_ids))
#assert arret le programme n cas une erreur 
# Le test doit être séparé de la KB
assert set(kb_ids).isdisjoint(set(test_ids))

# Le test doit être séparé de la validation
assert set(val_ids).isdisjoint(set(test_ids))

# Vérifier les doublons à l'intérieur de chaque ensemble
assert len(kb_ids) == len(set(kb_ids)) 
assert len(val_ids) == len(set(val_ids))
assert len(test_ids) == len(set(test_ids))

print("Split verification: OK")
print("Validation is a subset of KB.")
print("Test is independent from KB and validation.")

Split verification: OK
Validation is a subset of KB.
Test is independent from KB and validation.


images corrompues, captions < 5, doublons d’IDs

In [ ]:
bad_images = []
wrong_caps = []
empty_caps = []

hashes = {}
duplicate_images = []

modes = Counter()
caps_distribution = Counter()

for iid in tqdm(kb_ids):

    info = coco.loadImgs(iid)[0]
    path = f"data/images/train/{info['file_name']}"

    # Vérifier l'image
    try:
        with Image.open(path) as im:
            im.verify()

        with Image.open(path) as im:
            modes[im.mode] += 1

    except Exception:
        bad_images.append(iid)
        continue

    # Récupérer les captions
    anns = coco.loadAnns(
        coco.getAnnIds(imgIds=[iid])
    )

    n_caps = len(anns)
    caps_distribution[n_caps] += 1

    if n_caps != 5:
        wrong_caps.append((iid, n_caps))

    if any(not a["caption"].strip() for a in anns):
        empty_caps.append(iid)

    # Détecter les doublons exacts
    with open(path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    if file_hash in hashes:
        duplicate_images.append((iid, hashes[file_hash]))
    else:
        hashes[file_hash] = iid


print("\nKNOWLEDGE BASE QUALITY CHECK ")

print("Corrupted or missing images:", len(bad_images))
print("Images with a caption count different from 5:", len(wrong_caps))
print("Images with empty captions:", len(empty_caps))
print("Exact duplicate images:", len(duplicate_images))

print("Verified images:", len(hashes))
print("Image modes:", dict(modes))
print("Captions distribution:", dict(caps_distribution))

  0%|          | 0/10000 [00:00<?, ?it/s]

100%|██████████| 10000/10000 [00:31<00:00, 315.75it/s]


===== KNOWLEDGE BASE QUALITY CHECK =====
Corrupted or missing images: 0
Images with a caption count different from 5: 23
Images with empty captions: 0
Exact duplicate images: 0
Verified images: 10000
Image modes: {'RGB': 9984, 'L': 16}
Captions distribution: {5: 9977, 6: 22, 7: 1}


In [14]:
coco_val_caps = COCO(
    "data/annotations/captions_val2017.json"
)

coco_val_inst = COCO(
    "data/annotations/instances_val2017.json"
)

print("Test annotations loaded successfully.")

loading annotations into memory...
Done (t=0.28s)
creating index...
index created!
loading annotations into memory...
Done (t=0.39s)
creating index...
index created!
Test annotations loaded successfully.


In [15]:
test_bad = []
test_caps_count = Counter()
test_modes = Counter()

for iid in tqdm(test_ids):

    info = coco_val_caps.loadImgs(iid)[0]
    path = f"data/images/test/{info['file_name']}"

    # Vérifier l'intégrité de l'image
    try:
        with Image.open(path) as im:
            im.verify()

        with Image.open(path) as im:
            test_modes[im.mode] += 1

    except Exception:
        test_bad.append(iid)

    # Vérifier les captions
    anns = coco_val_caps.loadAnns(
        coco_val_caps.getAnnIds(imgIds=[iid])
    )

    test_caps_count[len(anns)] += 1


print("\n===== TEST QUALITY CHECK =====")

print("Corrupted or missing test images:", len(test_bad))
print("Captions distribution:", dict(test_caps_count))
print("Image modes:", dict(test_modes))

100%|██████████| 100/100 [00:00<00:00, 531.12it/s]


===== TEST QUALITY CHECK =====
Corrupted or missing test images: 0
Captions distribution: {5: 100}
Image modes: {'RGB': 99, 'L': 1}


In [16]:
test_caps = {}

for iid in test_ids:

    anns = coco_val_caps.loadAnns(
        coco_val_caps.getAnnIds(imgIds=[iid])
    )

    test_caps[str(iid)] = [
        a["caption"].strip() for a in anns
    ]


# Créer le dossier
os.makedirs("data/processed/test", exist_ok=True)

# Sauvegarder les captions
with open(
    "data/processed/test/test_captions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        test_caps,
        f,
        indent=4,
        ensure_ascii=False
    )


print("Test captions saved:", len(test_caps))
print("File: data/processed/test/test_captions.json")

Test captions saved: 100
File: data/processed/test/test_captions.json


In [ ]:
print("\nFINAL DATASET SUMMARY ")

print("Knowledge Base:", len(kb_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

print("\n SPLIT CHECK ")

print("Validation included in KB:",
      set(val_ids).issubset(set(kb_ids)))

print("Test separated from KB:",
      set(kb_ids).isdisjoint(set(test_ids)))

print("\nIMAGE QUALITY ")

print("KB corrupted images:", len(bad_images))
print("Test corrupted images:", len(test_bad))

print("KB images verified:", len(hashes))
print("Test captions saved:", len(test_caps))

print("\nCAPTIONS")

print("KB caption distribution:", dict(caps_distribution))
print("Test caption distribution:", dict(test_caps_count))


========== FINAL DATASET SUMMARY ==========
Knowledge Base: 10000
Validation: 100
Test: 100

========== SPLIT CHECK ==========
Validation included in KB: True
Test separated from KB: True

========== IMAGE QUALITY ==========
KB corrupted images: 0
Test corrupted images: 0
KB images verified: 10000
Test captions saved: 100

========== CAPTIONS ==========
KB caption distribution: {5: 9977, 6: 22, 7: 1}
Test caption distribution: {5: 100}


In [ ]:
print("Knowledge Base:", len(kb_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

print("\nOverlaps:")
print("Validation ∩ KB:", len(set(val_ids) & set(kb_ids)))
print("Test ∩ KB:", len(set(test_ids) & set(kb_ids)))
print("Validation ∩ Test:", len(set(val_ids) & set(test_ids)))

Knowledge Base: 10000
Validation: 100
Test: 100

Overlaps:
Validation ∩ KB: 100
Test ∩ KB: 0
Validation ∩ Test: 0
